# Getting started with LatticeQCD.jl v2 (Colab / Jupyter)

[LatticeQCD.jl](https://github.com/akio-tomiya/LatticeQCD.jl) is a lattice QCD package written in Julia. In this notebook we run simulations with the v2 workflow (`SimulationSpec` → `build_simulation` → `run!`) and plot the results.

- On Google Colab, the runtime is set to **Julia** automatically when you open this notebook (if not, choose Julia under Runtime → Change runtime type).
- Julia 1.11 and 1.12 are supported.
- Video (English): coming soon

## 1. Setup

Install LatticeQCD.jl and Plots.jl for plotting. The first run takes a few minutes.

In [ ]:
using Pkg
Pkg.add(["LatticeQCD", "Plots"])
VERSION

In [ ]:
using LatticeQCD, Plots, Statistics
pkgversion(LatticeQCD)

## 2. A first run: HMC with two flavors of Wilson fermions

The interactive `run_wizard()` is awkward in a notebook, so we write the settings as TOML and load them. They are close to the defaults of the wizard's simple mode (4⁴ lattice, β = 5.7, κ = 0.141139, cold start).

In [ ]:
write("hmc.toml", """
["Physical setting"]
L = [4, 4, 4, 4]
"β" = 5.7
update_method = "HMC"
Nwing = 1
Nthermalization = 1
Nsteps = 51

["Physical setting(fermions)"]
Dirac_operator = "Wilson"
quench = false
hop = 0.141139

["HMC related"]
"Δτ" = 0.05
MDsteps = 20

["System Control"]
log_dir = "./logs"
logfile = "hmc.txt"
measurement_basedir = "./measurements"
measurement_dir = "hmc"
hasgradientflow = false

["Measurement set".measurement_methods.Plaquette]
methodname = "Plaquette"
measure_every = 1
""")

spec = load_simulation_spec("hmc.toml")

Passing a `RecordingSimulationEventSink` stores every event of the run, including all measurements, so we can use them later in Julia. `verbose=false` turns off the per-trajectory printout.

In [ ]:
rec = RecordingSimulationEventSink()
session = build_simulation(spec, GaugefieldsEnvironment(); sink=rec)
@time result = run!(session; verbose=false)   # summary という名前は Julia の組み込み関数と重なるので使わない

In [ ]:
plaq = [(e.record.point.trajectory, e.record.value) for e in rec.events
        if e isa MeasurementFinished && e.record.name == :plaquette]
accepted = count(e -> e isa TrajectoryFinished && e.update.accepted, rec.events)
println("accepted: ", accepted, " / ", length(plaq))
plot(first.(plaq), last.(plaq), marker=:circle, xlabel="trajectory", ylabel="plaquette", legend=false)

Starting from a cold start, the plaquette begins at 1 and settles around 0.58 after thermalization. The results are also written to files (`measurements/` and `logs/`).

In [ ]:
readdir("measurements/hmc"), readdir("logs")

## 3. Measuring the pion mass (quenched approximation)

Gauge configurations are generated from gluons alone (heat bath with over-relaxation), and the pion correlator is measured with Wilson fermions (κ = 0.150). To finish in a few minutes on a Colab CPU, the lattice is a small 4³×16 and the quarks are rather heavy (κ = 0.150); the video uses 8³×16 and κ = 0.164.

In [ ]:
write("pion.toml", """
["Physical setting"]
L = [4, 4, 4, 16]
"β" = 5.7
update_method = "Heatbath"
useOR = true
numOR = 3
Nthermalization = 21
Nsteps = 141

["Physical setting(fermions)"]
Dirac_operator = "nothing"

["System Control"]
log_dir = "./logs"
logfile = "pion.txt"
measurement_basedir = "./measurements"
measurement_dir = "pion"
hasgradientflow = false

["Measurement set".measurement_methods.Pion_correlator]
methodname = "Pion_correlator"
measure_every = 4

    ["Measurement set".measurement_methods.Pion_correlator.fermion_parameters]
    Dirac_operator = "Wilson"
    hop = 0.150
""")

spec_pion = load_simulation_spec("pion.toml")
rec_pion = RecordingSimulationEventSink()
session_pion = build_simulation(spec_pion, GaugefieldsEnvironment(); sink=rec_pion)
@time run!(session_pion; verbose=false)

In [ ]:
C = [e.record.value for e in rec_pion.events
     if e isa MeasurementFinished && e.record.name == :pion_correlator]
Cm = mean(C)
println("number of measurements: ", length(C))
plot(0:length(Cm)-1, log.(Cm), marker=:circle, xlabel="t", ylabel="log C(t)", legend=false)

On a periodic lattice the correlator behaves as $C(t) \propto \cosh(m_\pi (t - N_t/2))$. We compute the effective mass $m_{\rm eff}(t) = \cosh^{-1}\frac{C(t-1)+C(t+1)}{2C(t)}$ from three neighboring time slices, and estimate errors with the jackknife method (leave one measurement out at a time).

In [ ]:
Nt = length(Cm)
meff(c) = [acosh(max(1, (c[t-1] + c[t+1]) / 2c[t])) for t in 2:Nt÷2+1]
n = length(C); S = sum(C)
jk = [meff((S - c) / (n - 1)) for c in C]
m = mean(jk); err = sqrt.((n - 1) / n * sum(x -> (x - m) .^ 2, jk))
scatter(1:Nt÷2, m, yerr=err, xlabel="t", ylabel="m_eff a", legend=false, ylims=(0, 2.5))

At small t heavier states contribute; at larger t the effective mass approaches a constant, the pion mass. Here the spatial size is only 4 and there are just 30 measurements, so the plateau (about 0.85–0.95 for t = 4–8) is noisy. A larger lattice (for example 8³×16) and more measurements make it clearer.

## Next steps

- README: https://github.com/akio-tomiya/LatticeQCD.jl
- Migrating from v1 to v2: `docs/src/migration-v2.md`
- Questions: [JuliaQCD discussions](https://github.com/orgs/JuliaQCD/discussions/2)